In [23]:
import sys
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

ROOT = Path("../../")
sys.path.insert(0, str(ROOT))

from src.fetch.yt_api import get_channel_video_ids, get_videos_metadata, Video

In [11]:
fringe_df = pd.read_csv(ROOT / "data/sourcing/fringe_media_channels.csv", sep=";")[["channel", "url", "id", "ideology"]].dropna()
fringe_df.head()

,channel,url,id,ideology
1,24nyt,https://www.youtube.com/@24nyt94,UCfwox5NVAQMIfvac164ok7A,Right
2,Arbejderen,https://www.youtube.com/@dagbladetarbejderen/,UCgdi47-cxS64ymUMOlvgJ7g,Left
3,Danmarks Frie Fjernsyn,https://www.youtube.com/@danmarksfriefjernsyn7842,UC71Vnm9HwHxOrJRk5U9-mlA,Anti-system
4,Den Korte Avis,https://www.youtube.com/@denkorteavis,UCcIhxy9IRp7tslhCSSvA24Q,Right
5,Dkdox.tv,https://www.youtube.com/@dkdoxtv,UC03g0H9JfNI97CRDSSvIs3w,Anti-system


In [10]:
video_ids = []

for idx, row in fringe_df.iterrows():
    vids : list[Video] = get_channel_video_ids(row["id"])
    for vid in tqdm(vids):
        video_ids.append({"channel": row["channel"], "video_id": vid.id})

video_df = pd.DataFrame(video_ids)

100%|██████████| 25/25 [00:00<00:00, 78899.62it/s]


In [24]:
video_metadata = []

video_list = video_df["video_id"].to_list()
video_list = [Video(id=vid) for vid in video_list]
chunks = [video_list[i:i + 50] for i in range(0, len(video_list), 50)]

for chunk in tqdm(chunks):
    vids = get_videos_metadata(chunk)
    for vid in vids:
        video_metadata.append({
            "video_id": vid.id,
            "channel": vid.channel,
            "title": vid.title,
            "description": vid.description,
            "tags": vid.tags,
            "category": vid.category
        })

video_metadata_df = pd.DataFrame(video_metadata)

100%|██████████| 24/24 [00:07<00:00,  3.39it/s]


In [25]:
mapping = {
    "24 NYT": "24nyt",
    "Dagbladet Arbejderen": "Arbejderen",
    "Danmarks Frie Fjernsyn": "Danmarks Frie Fjernsyn",
    "DEN KORTE AVIS": "Den Korte Avis",
    "dkdox .tv": "Dkdox.tv",
    "FOLKETS": "Folkets Avis",
    "Free Observer": "Free Observer",
    "Indblik": "Indblik",
    "Konfront": "Konfront",
    "Kontrast": "Kontrast",
    "NewSpeek": "NewSpeek.info",
    "Netavisen Pio": "Pio Pio",
    "Researchkollektivet Redox": "Redox",
    "Solidaritet": "Solidaritet"
}
video_metadata_df["channel"] = video_metadata_df["channel"].replace(mapping)

In [26]:
video_metadata_df = video_metadata_df.merge(fringe_df, on="channel", how="left")

In [27]:
video_metadata_df.head()

,video_id,channel,title,description,tags,category,url,id,ideology
0,nNgNwlop6hA,24nyt,Lars Mathiesen (NB) tordner mod bønnekald.,"Lars Mathiesen (NB) er rasende. Se her, hvorfor.",[],1,https://www.youtube.com/@24nyt94,UCfwox5NVAQMIfvac164ok7A,Right
1,KO9dvYaifZY,24nyt,Nytorv 010120,Efterlysning af gerningsmand.,[],1,https://www.youtube.com/@24nyt94,UCfwox5NVAQMIfvac164ok7A,Right
2,W2V--IqCFRk,24nyt,Falsk betjent taget på fersk gerning af ægte b...,Falsk betjent taget på fersk gerning af ægte b...,"[falsk betjent, politi]",1,https://www.youtube.com/@24nyt94,UCfwox5NVAQMIfvac164ok7A,Right
3,d5U8A3ka3Yw,24nyt,24nyt.dk Vold mod Rasmus Paludan i Jomfru Ane ...,,"[rasmus Paludan, 24nyt, overfald, vold, demons...",1,https://www.youtube.com/@24nyt94,UCfwox5NVAQMIfvac164ok7A,Right
4,j_6WyFV0lVY,24nyt,EU-vanvid,,"[EU, 24nyt]",1,https://www.youtube.com/@24nyt94,UCfwox5NVAQMIfvac164ok7A,Right


In [ ]:
video_metadata_df.to_csv(ROOT / "data/sourcing/out/fringe_media_videos.csv")